# LABR rating prediction — all remaining models (Groupe 3)
**Run order:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

- Everything is saved to Google Drive (`MyDrive/labr-results/`) **after each model**.
- If Colab disconnects: reconnect and **Run all** again → finished models are skipped automatically.
- First try `QUICK = True` (~10 min, tiny data) to check nothing crashes, then set `QUICK = False` for the real run (~2–3 h).
- Test metrics are computed **only in section F** (test opened once). Each model only *saves* its test predictions.

| Section | Models |
|---|---|
| A | Baselines again (A1 majority, A2 length, A3 TF-IDF+LR, A4 Ridge) → needed for test predictions |
| C | C1 BiLSTM · C2 BiLSTM+attention · C3 mini-Transformer (4 heads / 1 head / no positional encoding) |
| D | Frozen AraBERT features → Logistic Regression (CLS vs mean pooling) |
| E | Fine-tuned AraBERT: E1 classification · E2 regression · E3 multi-task |
| F | Test: CIs, unseen books/users, perturbations, McNemar |
| G | Final table: quality vs cost vs latency |

In [ ]:
# ===== 0. CONFIG =====
QUICK = False          # True = tiny smoke test (~10 min). False = real run.
SEED = 42

import sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabert"], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/labr-results")
else:
    BASE = Path("labr-results")
OUT = BASE / ("quick" if QUICK else "full")
DATA = BASE / "data"
for d in ["results", "preds", "attention", "figures"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

EPOCHS_SMALL = 2 if QUICK else 8     # C models (early stopping, patience 2)
EPOCHS_BERT = 1 if QUICK else 3      # E models
print("QUICK:", QUICK, "| outputs ->", OUT)

In [ ]:
# ===== 1. UTILITIES (data, normalization, metrics, saving) =====
import re, csv, json, time, math, random, urllib.request
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error, confusion_matrix

BASE_URL = "https://raw.githubusercontent.com/mohamedadaly/LABR/master/data/"
FILES = ["reviews.tsv", "5class-unbalanced-train.txt", "5class-unbalanced-test.txt"]
COLS = ["rating", "review_id", "user_id", "book_id", "text"]
DIAC = re.compile(r'[ؗ-ًؚ-ْٰ]')

def download_labr(data_dir):
    data_dir.mkdir(parents=True, exist_ok=True)
    for f in FILES:
        dest = data_dir / f
        if dest.exists(): continue
        tmp = dest.with_suffix(dest.suffix + ".part")
        with urllib.request.urlopen(BASE_URL + f, timeout=60) as r, open(tmp, "wb") as out:
            while chunk := r.read(1 << 20): out.write(chunk)
        tmp.rename(dest)

def normalize(t):
    t = DIAC.sub('', str(t)).replace('ـ', '')
    t = re.sub('[إأآا]', 'ا', t).replace('ى', 'ي').replace('ة', 'ه')
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)
    return re.sub(r'\s+', ' ', t).strip()

def make_splits(df, tr_ids, te_ids, val_size=0.1, seed=42):
    df = df.assign(row=np.arange(len(df)))
    train, test = df.iloc[tr_ids], df.iloc[te_ids]
    train = train[~train.norm.isin(set(test.norm))]                       # remove leakage
    cnt = train.groupby(["norm", "rating"]).size().rename("c").reset_index()
    cnt["is_max"] = cnt.c == cnt.groupby("norm").c.transform("max")
    n_max = cnt[cnt.is_max].groupby("norm").size()
    winners = cnt[cnt.is_max & cnt.norm.isin(n_max[n_max == 1].index)][["norm", "rating"]]
    train = train.merge(winners, on=["norm", "rating"]).drop_duplicates("norm")   # dedupe + majority label
    train, val = train_test_split(train, test_size=val_size, stratify=train.rating, random_state=seed)
    return train, val, test

def evaluate(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"macro_f1": round(f1_score(y, p, average="macro"), 4),
            "accuracy": round(accuracy_score(y, p), 4),
            "mae": round(mean_absolute_error(y, p), 4)}

def confusion(y, p):
    L = [1, 2, 3, 4, 5]
    return pd.DataFrame(confusion_matrix(y, p, labels=L), index=[f"true {l}" for l in L], columns=[f"pred {l}" for l in L])

def done(name):
    return (OUT / "results" / f"{name}.json").exists()

def save_model_outputs(name, outs, info):
    """outs = {split_name: (df, y_pred, probs_or_None)}. Saves predictions + val metrics. Test metrics come in section F."""
    for split, (d, pred, prob) in outs.items():
        o = pd.DataFrame({"row": d.row.values, "y_true": d.rating.values, "y_pred": np.asarray(pred).astype(int)})
        if prob is not None:
            for k in range(5): o[f"p{k+1}"] = np.asarray(prob)[:, k]
        o.to_csv(OUT / "preds" / f"{name}__{split}.csv", index=False)
    d, pred, _ = outs["val"]
    info = {"model": name, **{f"val_{k}": v for k, v in evaluate(d.rating, pred).items()}, **info}
    json.dump(info, open(OUT / "results" / f"{name}.json", "w"), indent=1, default=str)
    print({k: v for k, v in info.items() if k != "history"})

def latency_ms(fn, n=100):
    """average time to predict ONE review (batch size 1), in milliseconds"""
    fn(0)                                   # warm-up
    t0 = time.perf_counter()
    for i in range(n): fn(i)
    return round((time.perf_counter() - t0) / n * 1000, 2)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass

# --- perturbations for the robustness test (section F) ---
def pert_noexcl(t, rng):  return t.replace("!", "")                                 # remove all "!"
def pert_tatweel(t, rng): return " ".join(w[:len(w)//2] + "ـ" + w[len(w)//2:] if len(w) > 2 and rng.random() < 0.3 else w for w in t.split())
def pert_elong(t, rng):   return " ".join(w + w[-1] * 3 if len(w) > 2 and rng.random() < 0.2 else w for w in t.split())
PERTURBATIONS = {"noexcl": pert_noexcl, "tatweel": pert_tatweel, "elong": pert_elong}

In [ ]:
# ===== 2. DATA (same splits as Phase 1, rebuilt deterministically) =====
if not (DATA / "splits" / "val.csv").exists():
    download_labr(DATA)
    df = pd.read_csv(DATA / "reviews.tsv", sep="\t", header=None, names=COLS, quoting=csv.QUOTE_NONE, encoding="utf-8")
    df["text"] = df.text.fillna("").astype(str); df["norm"] = df.text.map(normalize)
    tr_ids = np.loadtxt(DATA / "5class-unbalanced-train.txt", dtype=int)
    te_ids = np.loadtxt(DATA / "5class-unbalanced-test.txt", dtype=int)
    (DATA / "splits").mkdir(exist_ok=True)
    for n, d in zip(["train", "val", "test"], make_splits(df, tr_ids, te_ids)):
        d.to_csv(DATA / "splits" / f"{n}.csv", index=False, encoding="utf-8")

S = {n: pd.read_csv(DATA / "splits" / f"{n}.csv", encoding="utf-8", keep_default_na=False) for n in ["train", "val", "test"]}
train, val, test = S["train"], S["val"], S["test"]
if QUICK:
    train = train.sample(3000, random_state=SEED); val = val.sample(600, random_state=SEED); test = test.sample(1000, random_state=SEED)
train, val, test = (d.reset_index(drop=True) for d in (train, val, test))
y_tr, y_va = train.rating.values, val.rating.values

rng = random.Random(SEED)
TEST_PERT = {k: [f(t, rng) for t in test.text] for k, f in PERTURBATIONS.items()}   # raw perturbed test texts
print(len(train), len(val), len(test))

## A — Baselines (retrained to get test predictions)

In [ ]:
# ===== A. BASELINES =====
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

def all_splits(predict):          # predict(list_of_texts_raw) -> (pred, prob)
    outs = {"val": (val, *predict(list(val.text))), "test": (test, *predict(list(test.text)))}
    for k, texts in TEST_PERT.items(): outs[f"test_{k}"] = (test, *predict(texts))
    return outs

if not done("A1_majority"):
    m = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_tr), 1)), y_tr)
    pr = lambda T: (m.predict(np.zeros((len(T), 1))), None)
    save_model_outputs("A1_majority", all_splits(pr), {"n_params": 0, "train_time_min": 0, "latency_ms": 0})

if not done("A2_length"):
    lf = lambda T: np.c_[np.log1p([len(t.split()) for t in T]), np.log1p([len(t) for t in T])]
    t0 = time.time(); m = LogisticRegression(class_weight="balanced", max_iter=1000).fit(lf(train.text), y_tr); tt = time.time() - t0
    pr = lambda T: (m.predict(lf(T)), m.predict_proba(lf(T)))
    save_model_outputs("A2_length", all_splits(pr), {"n_params": 6, "train_time_min": round(tt/60, 2),
                       "latency_ms": latency_ms(lambda i: pr([val.text[i]]))})

if not (done("A3_tfidf_LR") and done("A4_tfidf_ridge")):
    t0 = time.time()
    vec_w = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000)
    X_tr = hstack([vec_w.fit_transform(train.norm), vec_c.fit_transform(train.norm)]).tocsr()
    feats = lambda T: hstack([vec_w.transform([normalize(t) for t in T]), vec_c.transform([normalize(t) for t in T])]).tocsr()
    t_feat = time.time() - t0

    if not done("A3_tfidf_LR"):
        t0 = time.time(); lr = LogisticRegression(C=1, class_weight="balanced", max_iter=2000).fit(X_tr, y_tr); tt = time.time() - t0 + t_feat
        pr = lambda T: (lr.predict(feats(T)), lr.predict_proba(feats(T)))
        save_model_outputs("A3_tfidf_LR", all_splits(pr), {"n_params": lr.coef_.size, "train_time_min": round(tt/60, 2),
                           "latency_ms": latency_ms(lambda i: pr([val.text[i]]))})
    if not done("A4_tfidf_ridge"):
        t0 = time.time(); rg = Ridge(alpha=1).fit(X_tr, y_tr); tt = time.time() - t0 + t_feat
        pr = lambda T: (np.clip(np.rint(rg.predict(feats(T))), 1, 5), None)
        save_model_outputs("A4_tfidf_ridge", all_splits(pr), {"n_params": rg.coef_.size, "train_time_min": round(tt/60, 2),
                           "latency_ms": latency_ms(lambda i: pr([val.text[i]]))})

## C — Neural models from scratch (shared code)

In [ ]:
# ===== C0. SHARED NEURAL CODE: vocabulary, encoding, training loop =====
import torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV)

TOK_RE = re.compile(r"\w+|[!؟?]")              # words + "!" "?" "؟" kept as tokens (intensity)
tokenize = lambda t: TOK_RE.findall(t)
cnt = Counter(w for t in train.norm for w in tokenize(t))
itos = ["<pad>", "<unk>", "<cls>"] + [w for w, c in cnt.most_common(50_000) if c >= 2]   # vocab from TRAIN only
stoi = {w: i for i, w in enumerate(itos)}
MAX_LEN = 150                                  # ~90% of reviews have <=139 words

def encode(texts_norm):
    X = np.zeros((len(texts_norm), MAX_LEN), dtype=np.int64)       # 0 = padding
    for i, t in enumerate(texts_norm):
        ids = [2] + [stoi.get(w, 1) for w in tokenize(t)][:MAX_LEN - 1]   # 2 = <cls> first, 1 = unknown word
        X[i, :len(ids)] = ids
    return torch.tensor(X)

Xtr, Xva, Xte = encode(train.norm), encode(val.norm), encode(test.norm)
Xte_pert = {k: encode([normalize(t) for t in T]) for k, T in TEST_PERT.items()}
freq = np.bincount(y_tr, minlength=6)[1:]
CW = torch.tensor(len(y_tr) / (5 * freq), dtype=torch.float, device=DEV)   # class weights: rare class -> bigger weight
print("vocab:", len(itos), "| class weights:", CW.cpu().numpy().round(2))

@torch.no_grad()
def predict_logits(model, X, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = X[i:i+bs].to(DEV); out.append(model(xb, xb != 0)[0].float().cpu())
    return torch.cat(out)

def train_small(model, epochs, lr, bs=64, patience=2):
    model.to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    ytr_t = torch.tensor(y_tr - 1)                      # classes 0..4
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr)); tl = 0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i+bs]; xb, yb = Xtr[idx].to(DEV), ytr_t[idx].to(DEV)
            loss = F.cross_entropy(model(xb, xb != 0)[0], yb, weight=CW)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * len(idx)
        f1 = f1_score(y_va, predict_logits(model, Xva).argmax(1).numpy() + 1, average="macro")
        hist.append({"epoch": ep + 1, "train_loss": round(tl / len(Xtr), 4), "val_f1": round(f1, 4)}); print(hist[-1])
        if f1 > best: best, bad, best_state = f1, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break                    # early stopping
    model.load_state_dict(best_state); return model, hist

def run_small(name, make_model, lr):
    if done(name): print("skip", name); return None
    set_seed(); model = make_model()
    t0 = time.time(); model, hist = train_small(model, EPOCHS_SMALL, lr); tt = time.time() - t0
    outs = {}
    for split, d, X in [("val", val, Xva), ("test", test, Xte)] + [(f"test_{k}", test, Xp) for k, Xp in Xte_pert.items()]:
        P = predict_logits(model, X).softmax(1).numpy(); outs[split] = (d, P.argmax(1) + 1, P)
    save_model_outputs(name, outs, {"n_params": sum(p.numel() for p in model.parameters()), "train_time_min": round(tt/60, 2),
                       "latency_ms": latency_ms(lambda i: predict_logits(model, Xva[i:i+1])), "history": hist})
    return model

@torch.no_grad()
def save_attention(name, model, n=30):
    """saves tokens + attention weights of n val reviews, for interpretation"""
    if model is None: return
    model.eval(); X = Xva[:n].to(DEV); _, att = model(X, X != 0); recs = []
    for i in range(n):
        L = int((X[i] != 0).sum())
        recs.append({"rating": int(y_va[i]), "tokens": [itos[j] for j in X[i, :L].tolist()],
                     "attention": att[i][..., :L].float().cpu().numpy().round(4).tolist()})
    json.dump(recs, open(OUT / "attention" / f"{name}.json", "w"), ensure_ascii=False)

In [ ]:
# ===== C1 / C2. BiLSTM (pooling) and BiLSTM + additive attention =====
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTM(nn.Module):
    def __init__(s, V, emb=200, hid=128, attention=False, drop=0.3):
        super().__init__()
        s.emb = nn.Embedding(V, emb, padding_idx=0)
        s.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        s.attention, s.drop = attention, nn.Dropout(drop)
        if attention:
            s.W = nn.Linear(2 * hid, 2 * hid); s.v = nn.Linear(2 * hid, 1, bias=False)   # score_t = v·tanh(W h_t)
            s.fc = nn.Linear(2 * hid, 5)
        else:
            s.fc = nn.Linear(4 * hid, 5)                                                # [mean ; max] pooling
    def forward(s, x, mask):
        lengths = mask.sum(1).cpu()
        packed = pack_padded_sequence(s.drop(s.emb(x)), lengths, batch_first=True, enforce_sorted=False)
        h, _ = pad_packed_sequence(s.lstm(packed)[0], batch_first=True, total_length=x.size(1))   # (B,T,2H)
        if s.attention:
            e = s.v(torch.tanh(s.W(h))).squeeze(-1).masked_fill(~mask, -1e9)   # one score per word
            a = e.softmax(-1)                                                   # attention weights, sum to 1
            return s.fc(s.drop((a.unsqueeze(-1) * h).sum(1))), a                # weighted sum of word states
        m = mask.unsqueeze(-1).float()
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        mx = h.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values
        return s.fc(s.drop(torch.cat([mean, mx], -1))), None

V = len(itos)
run_small("C1_bilstm", lambda: BiLSTM(V), lr=1e-3)
save_attention("C2_bilstm_attention", run_small("C2_bilstm_attention", lambda: BiLSTM(V, attention=True), lr=1e-3))

In [ ]:
# ===== C3. MINI-TRANSFORMER FROM SCRATCH (self-attention, multi-head, positional encoding) =====
class MultiHeadSelfAttention(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.h, s.dk = h, d // h
        s.qkv = nn.Linear(d, 3 * d)          # one projection producing Q, K, V
        s.o = nn.Linear(d, d)                # recombines the heads
    def forward(s, x, mask):
        B, T, D = x.shape
        q, k, v = s.qkv(x).view(B, T, 3, s.h, s.dk).permute(2, 0, 3, 1, 4)    # each: (B, heads, T, dk)
        scores = q @ k.transpose(-2, -1) / math.sqrt(s.dk)                     # (B, heads, T, T)
        scores = scores.masked_fill(~mask[:, None, None, :], -1e9)            # never attend to padding
        att = scores.softmax(-1)
        out = (att @ v).transpose(1, 2).reshape(B, T, D)                       # concat heads
        return s.o(out), att

class Block(nn.Module):
    def __init__(s, d, h, ff, drop):
        super().__init__()
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn = MultiHeadSelfAttention(d, h)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        s.drop = nn.Dropout(drop)
    def forward(s, x, mask):
        a, att = s.attn(s.ln1(x), mask); x = x + s.drop(a)          # residual connection 1
        x = x + s.drop(s.ff(s.ln2(x)))                               # residual connection 2
        return x, att

class MiniTransformer(nn.Module):
    def __init__(s, V, d=128, heads=4, layers=2, ff=256, pos=True, drop=0.1):
        super().__init__(); s.d, s.pos = d, pos
        s.emb = nn.Embedding(V, d, padding_idx=0)
        pe = torch.zeros(MAX_LEN, d); p = torch.arange(MAX_LEN).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(p * div), torch.cos(p * div)       # sinusoidal positional encoding
        s.register_buffer("pe", pe)
        s.blocks = nn.ModuleList([Block(d, heads, ff, drop) for _ in range(layers)])
        s.ln, s.fc, s.drop = nn.LayerNorm(d), nn.Linear(d, 5), nn.Dropout(drop)
    def forward(s, x, mask):
        h = s.emb(x) * math.sqrt(s.d)
        if s.pos: h = h + s.pe[: x.size(1)]
        h = s.drop(h)
        for b in s.blocks: h, att = b(h, mask)
        return s.fc(s.ln(h)[:, 0]), att[:, :, 0, :]      # classify from <cls>; return what <cls> attends to (last layer)

save_attention("C3_transformer_4heads", run_small("C3_transformer_4heads", lambda: MiniTransformer(V, heads=4), lr=5e-4))
save_attention("C3_transformer_1head",  run_small("C3_transformer_1head",  lambda: MiniTransformer(V, heads=1), lr=5e-4))
run_small("C3_transformer_noPE", lambda: MiniTransformer(V, heads=4, pos=False), lr=5e-4)

## D / E — AraBERT (shared code)

In [ ]:
# ===== DE0. AraBERT tokenizer, preprocessing, batching =====
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
BERT = "aubmindlab/bert-base-arabertv02"
BERT_MAXLEN = 256                             # ~97% of reviews fit entirely
try:
    from arabert.preprocess import ArabertPreprocessor
    prep = ArabertPreprocessor(model_name=BERT).preprocess
except Exception as e:
    print("AraBERT preprocessor unavailable, using raw text:", e); prep = lambda t: t

tok = AutoTokenizer.from_pretrained(BERT)
enc_ids = lambda T: tok([prep(t) for t in T], truncation=True, max_length=BERT_MAXLEN)["input_ids"]
ids_tr, ids_va, ids_te = enc_ids(train.text), enc_ids(val.text), enc_ids(test.text)
ids_pert = {k: enc_ids(T) for k, T in TEST_PERT.items()}
print("share truncated (val):", np.mean([len(s) >= BERT_MAXLEN for s in ids_va]).round(3))

def pad_batch(seqs):
    x = torch.full((len(seqs), max(map(len, seqs))), tok.pad_token_id)
    for i, s in enumerate(seqs): x[i, :len(s)] = torch.tensor(s)
    return x.to(DEV), (x != tok.pad_token_id).long().to(DEV)

@torch.no_grad()
def run_batches(fn, ids, bs=64):
    """applies fn(x, mask) -> (B, k) on length-sorted batches (fast), returns results in original order"""
    order = np.argsort([len(s) for s in ids]); out = [None] * len(ids)
    for i in range(0, len(ids), bs):
        idx = order[i:i+bs]; r = fn(*pad_batch([ids[j] for j in idx])).float().cpu()
        for j, row in zip(idx, r): out[j] = row
    return torch.stack(out)

AMP = DEV.type == "cuda"     # mixed precision (fp16) on GPU: ~2x faster

In [ ]:
# ===== D. FROZEN AraBERT -> Logistic Regression (CLS vs mean pooling) =====
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

if not (done("D_frozen_cls") and done("D_frozen_mean")):
    enc = AutoModel.from_pretrained(BERT).to(DEV).eval()
    def feats(x, m):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            h = enc(input_ids=x, attention_mask=m).last_hidden_state.float()
        cls = h[:, 0]                                                     # [CLS] vector
        mean = (h * m.unsqueeze(-1)).sum(1) / m.sum(1, keepdim=True)      # average of all real tokens
        return torch.cat([cls, mean], -1)
    t0 = time.time()
    Ftr, Fva, Fte = (run_batches(feats, I).numpy() for I in (ids_tr, ids_va, ids_te))
    Fpert = {k: run_batches(feats, I).numpy() for k, I in ids_pert.items()}
    t_feat = time.time() - t0
    H = enc.config.hidden_size
    for name, sl in [("D_frozen_cls", slice(0, H)), ("D_frozen_mean", slice(H, 2 * H))]:
        if done(name): continue
        t0 = time.time(); best = None
        for C in [0.1, 1.0]:                                               # C chosen on val
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=3000)).fit(Ftr[:, sl], y_tr)
            f1 = f1_score(y_va, clf.predict(Fva[:, sl]), average="macro"); print(name, "C", C, round(f1, 4))
            if best is None or f1 > best[0]: best = (f1, C, clf)
        clf = best[2]; tt = time.time() - t0 + t_feat
        outs = {"val": (val, clf.predict(Fva[:, sl]), clf.predict_proba(Fva[:, sl])),
                "test": (test, clf.predict(Fte[:, sl]), clf.predict_proba(Fte[:, sl]))}
        for k, Fp in Fpert.items(): outs[f"test_{k}"] = (test, clf.predict(Fp[:, sl]), clf.predict_proba(Fp[:, sl]))
        lat = latency_ms(lambda i: clf.predict(run_batches(feats, ids_va[i:i+1]).numpy()[:, sl]))
        save_model_outputs(name, outs, {"n_params": sum(p.numel() for p in enc.parameters()) + clf[-1].coef_.size,
                           "trainable_params": clf[-1].coef_.size, "best_C": best[1], "train_time_min": round(tt/60, 2), "latency_ms": lat})
    del enc; torch.cuda.empty_cache()

In [ ]:
# ===== E. FINE-TUNED AraBERT: E1 classification, E2 regression, E3 multi-task =====
class BertRater(nn.Module):
    def __init__(s):
        super().__init__()
        s.bert = AutoModel.from_pretrained(BERT); H = s.bert.config.hidden_size
        s.drop, s.cls_head, s.reg_head = nn.Dropout(0.1), nn.Linear(H, 5), nn.Linear(H, 1)
    def forward(s, x, m):
        h = s.drop(s.bert(input_ids=x, attention_mask=m).last_hidden_state[:, 0])    # [CLS] vector
        return s.cls_head(h), s.reg_head(h).squeeze(-1) + 3                           # 5 logits, 1 star value (centered at 3)

def bert_outputs(model, ids):
    model.eval()
    def fn(x, m):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            lc, lr_ = model(x, m)
        return torch.cat([lc.float().softmax(-1), lr_.float().unsqueeze(-1)], -1)
    R = run_batches(fn, ids).numpy()
    return R[:, :5], R[:, 5]                                   # class probabilities, regression value

def to_preds(P, r, head):
    return P.argmax(1) + 1 if head == "cls" else np.clip(np.rint(r), 1, 5).astype(int)

def train_bert(mode, epochs=EPOCHS_BERT, lr=2e-5, bs=32):
    set_seed(); model = BertRater().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_steps = epochs * math.ceil(len(ids_tr) / bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * n_steps), n_steps)   # warm-up 10%, then linear decay
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    yc, yr = torch.tensor(y_tr - 1), torch.tensor(y_tr, dtype=torch.float)
    lens = [len(s) for s in ids_tr]; head = "reg" if mode == "reg" else "cls"
    best, best_state, hist = None, None, []
    for ep in range(epochs):
        model.train(); perm = np.random.permutation(len(ids_tr)); batches = []
        for c in range(0, len(perm), bs * 50):                     # group similar lengths -> less padding -> faster
            chunk = sorted(perm[c:c + bs * 50], key=lambda j: lens[j])
            batches += [chunk[i:i + bs] for i in range(0, len(chunk), bs)]
        random.shuffle(batches); tl = 0
        for b in tqdm(batches, desc=f"{mode} ep{ep+1}", mininterval=60):
            x, m = pad_batch([ids_tr[j] for j in b]); bi = torch.as_tensor(b); tc, tr_ = yc[bi].to(DEV), yr[bi].to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                lc, lr_ = model(x, m)
            ce = F.cross_entropy(lc.float(), tc, weight=CW); mse = F.mse_loss(lr_.float(), tr_)
            loss = ce if mode == "cls" else mse if mode == "reg" else ce + 0.5 * mse
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(b)
        P, r = bert_outputs(model, ids_va); ev = evaluate(y_va, to_preds(P, r, head))
        hist.append({"epoch": ep + 1, "train_loss": round(tl / len(ids_tr), 4), **ev}); print(hist[-1])
        score = -ev["mae"] if mode == "reg" else ev["macro_f1"]      # selection on val: MAE for regression, F1 otherwise
        if best is None or score > best:
            best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); return model, hist

for name, mode, heads in [("E1_bert_cls", "cls", ["cls"]), ("E2_bert_reg", "reg", ["reg"]), ("E3_bert_multitask", "multi", ["cls", "reg"])]:
    if done(name): print("skip", name); continue
    t0 = time.time(); model, hist = train_bert(mode); tt = time.time() - t0
    raw = {"val": (val, *bert_outputs(model, ids_va)), "test": (test, *bert_outputs(model, ids_te))}
    for k, I in ids_pert.items(): raw[f"test_{k}"] = (test, *bert_outputs(model, I))
    lat = latency_ms(lambda i: bert_outputs(model, ids_va[i:i+1]))
    n_params = sum(p.numel() for p in model.parameters())
    for h in heads:                     # multi-task model is evaluated with both heads
        nm = name if len(heads) == 1 else f"{name}_{h}head"
        outs = {s: (d, to_preds(P, r, h), P if h == "cls" else None) for s, (d, P, r) in raw.items()}
        save_model_outputs(nm, outs, {"n_params": n_params, "train_time_min": round(tt/60, 2), "latency_ms": lat, "history": hist})
    json.dump({}, open(OUT / "results" / f"{name}.json", "w")) if len(heads) > 1 else None   # marker so it's skipped on rerun
    del model; torch.cuda.empty_cache()

## F — Test set, opened once: CIs, unseen books/users, perturbations, McNemar

In [ ]:
# ===== F. ROBUSTNESS ON TEST =====
from scipy.stats import chi2
names = sorted(p.stem for p in (OUT / "results").glob("*.json") if json.load(open(p)))   # skip empty markers
seen_book, seen_user = set(train.book_id), set(train.user_id)
unseen_book = ~test.book_id.isin(seen_book).values
unseen_user = ~test.user_id.isin(seen_user).values
print(f"test reviews on books never seen in train: {unseen_book.mean():.1%} | by users never seen: {unseen_user.mean():.1%}")

def load_pred(name, split):
    p = pd.read_csv(OUT / "preds" / f"{name}__{split}.csv")
    assert (p.row.values == test.row.values).all(); return p.y_pred.values

def boot_ci(y, p, B=1000):
    r = np.random.default_rng(SEED)                      # same resamples for every model -> fair comparison
    s = [f1_score(y[i], p[i], average="macro") for i in (r.integers(0, len(y), len(y)) for _ in range(B))]
    return round(np.percentile(s, 2.5), 4), round(np.percentile(s, 97.5), 4)

y_te = test.rating.values; rows = []
for n in names:
    p = load_pred(n, "test"); ev = evaluate(y_te, p); lo, hi = boot_ci(y_te, p)
    row = {"model": n, "test_macro_f1": ev["macro_f1"], "ci95_low": lo, "ci95_high": hi, "test_accuracy": ev["accuracy"], "test_mae": ev["mae"],
           "f1_seen_books": round(f1_score(y_te[~unseen_book], p[~unseen_book], average="macro"), 4),
           "f1_unseen_books": round(f1_score(y_te[unseen_book], p[unseen_book], average="macro"), 4),
           "f1_seen_users": round(f1_score(y_te[~unseen_user], p[~unseen_user], average="macro"), 4),
           "f1_unseen_users": round(f1_score(y_te[unseen_user], p[unseen_user], average="macro"), 4)}
    for k in PERTURBATIONS:
        row[f"drop_{k}"] = round(f1_score(y_te, load_pred(n, f"test_{k}"), average="macro") - ev["macro_f1"], 4)   # negative = got worse
    rows.append(row)
F_table = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
F_table.to_csv(OUT / "results" / "F_test_robustness.csv", index=False)

def mcnemar(a, b):
    ca, cb = load_pred(a, "test") == y_te, load_pred(b, "test") == y_te
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())               # disagreements only
    stat = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
    return {"A": a, "B": b, "A_right_B_wrong": n01, "A_wrong_B_right": n10, "chi2": round(stat, 3), "p_value": float(chi2.sf(stat, 1))}

top = F_table.model.tolist()
pairs = [(top[0], top[1])] + ([(top[0], "A3_tfidf_LR")] if "A3_tfidf_LR" in top and top[0] != "A3_tfidf_LR" else [])
MC = pd.DataFrame([mcnemar(a, b) for a, b in pairs]); MC.to_csv(OUT / "results" / "F_mcnemar.csv", index=False)
print("best on test:", top[0]); print(confusion(y_te, load_pred(top[0], "test")))
display(F_table); display(MC)

## G — Final table: quality vs cost vs latency

In [ ]:
# ===== G. FINAL TABLE =====
import matplotlib.pyplot as plt
info = pd.DataFrame([json.load(open(OUT / "results" / f"{n}.json")) for n in names]).drop(columns=["history"], errors="ignore")
G = F_table.merge(info, on="model")
cols = ["model", "val_macro_f1", "test_macro_f1", "ci95_low", "ci95_high", "test_mae", "f1_unseen_books", "n_params", "train_time_min", "latency_ms"]
G = G[[c for c in cols if c in G]].sort_values("test_macro_f1", ascending=False)
G.to_csv(OUT / "results" / "G_final_table.csv", index=False)
display(G)

fig, ax = plt.subplots(figsize=(9, 5))
for _, r in G.iterrows():
    ax.scatter(max(r.latency_ms, 0.01), r.test_macro_f1)
    ax.annotate(r.model, (max(r.latency_ms, 0.01), r.test_macro_f1), fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("latency per review (ms, log scale)"); ax.set_ylabel("test macro-F1")
ax.set_title("Quality vs latency"); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / "figures" / "G_quality_vs_latency.png", dpi=150); plt.show()

import shutil
shutil.make_archive(str(BASE / f"labr_outputs_{'quick' if QUICK else 'full'}"), "zip", OUT)
print("ALL DONE ->", BASE)